# SetFit — Classification de texte few-shot (émotions)

Entraînement few-shot avec **SetFit** sur le dataset `dair-ai/emotion` (6 émotions, en lien avec la psychologie).

On fait varier le **nombre d'échantillons par classe** (8, 10, 20, 50, 100) et le **nombre d'epochs** (1, 5, 10), puis on évalue chaque modèle sur le **même jeu de test** avec accuracy, precision, recall et F1.

⚠️ **Active le GPU** : *Exécution → Modifier le type d'exécution → T4 GPU*. Sinon l'entraînement des 15 modèles sera très lent.


## 1. Installation

In [ ]:
!pip install -q "setfit==1.1.0" "sentence-transformers==3.3.1" "transformers==4.45.2" "datasets<3.0.0" scikit-learn pandas matplotlib
print("Installé.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 6.7 MB/s eta 0:00:00
Installé.


## 2. Vérifier le GPU

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Carte :", torch.cuda.get_device_name(0))

GPU disponible : True
Carte : Tesla T4


## 3. Préparer les données (split 80 / 20)

In [ ]:
from datasets import load_dataset, concatenate_datasets

DATASET = "dair-ai/emotion"
SEED = 42
TEST_SIZE = 0.20
TEST_CAP = 500

ds = load_dataset(DATASET)
full = concatenate_datasets([ds["train"], ds["validation"], ds["test"]])
split = full.train_test_split(test_size=TEST_SIZE, seed=SEED, stratify_by_column="label")
train_pool = split["train"]
test_set = split["test"].shuffle(seed=SEED).select(range(min(TEST_CAP, len(split["test"]))))

test_texts = test_set["text"]
test_labels = list(test_set["label"])
print(f"Pool d'entraînement : {len(train_pool)}  |  Test fixe : {len(test_set)}")
print("Classes :", ds["train"].features["label"].names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Pool d'entraînement : 16000  |  Test fixe : 500
Classes : ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


## 4. Lancer la grille d'expériences (5 × 3 = 15 modèles)

Cela prend plusieurs minutes sur GPU. Patiente.

In [ ]:
import time, pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset

BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # rapide
SAMPLES_PER_CLASS = [8, 10, 20, 50, 100]
EPOCHS = [1, 5, 10]
BATCH_SIZE = 16

results = []
total = len(SAMPLES_PER_CLASS) * len(EPOCHS)
i = 0
for n in SAMPLES_PER_CLASS:
    for e in EPOCHS:
        i += 1
        print(f"\n[{i}/{total}] {n} échantillons/classe, {e} epochs...")
        train_few = sample_dataset(train_pool, label_column="label", num_samples=n)
        model = SetFitModel.from_pretrained(BASE_MODEL)
        args = TrainingArguments(batch_size=BATCH_SIZE, num_epochs=e)
        trainer = Trainer(model=model, args=args, train_dataset=train_few)
        t0 = time.time()
        trainer.train()
        dt = time.time() - t0
        preds = [int(p) for p in model.predict(test_texts)]
        acc = accuracy_score(test_labels, preds)
        prec, rec, f1, _ = precision_recall_fscore_support(test_labels, preds, average="macro", zero_division=0)
        results.append({"samples_per_class": n, "epochs": e, "train_examples": len(train_few),
                        "accuracy": round(acc,4), "precision": round(prec,4),
                        "recall": round(rec,4), "f1": round(f1,4), "train_time_s": round(dt,1)})
        print(f"   acc={acc:.3f}  P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}  ({dt:.0f}s)")

df = pd.DataFrame(results)
df.to_csv("results_setfit.csv", index=False)
print("\nTerminé. Résultats dans results_setfit.csv")
df


[1/15] 8 échantillons/classe, 1 epochs...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/datasets/utils/_dill.py:385: DeprecationWarning: co_lnotab is deprecated, use co_lines instead.
  obj.co_lnotab,  # for < python 3.10 [not counted in args]
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.ut

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 1920
  Batch size = 16
  Num epochs = 1
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.12/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: A

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:269: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  app_url = wandb.util.app_url(tags["base_url"])  # type: ignore[index]
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:280: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


...

Step,Training Loss
1,0.523400
50,0.344300
100,0.329000
150,0.340200
200,0.315600
250,0.309900
300,0.304800
350,0.288000
400,0.311800
450,0.284800


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/py

Step,Training Loss
1,0.523400
50,0.344300
100,0.329000
150,0.340200
200,0.315600
250,0.309900
300,0.304800
350,0.288000
400,0.311800
450,0.284800


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/py

## 5. Tableau récapitulatif

In [2]:
import pandas as pd
pd.read_csv("results_setfit.csv")

## 6. Visualisations

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.read_csv("results_setfit.csv")
pivot = df.pivot(index="samples_per_class", columns="epochs", values="f1")

fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(pivot.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel("Epochs"); ax.set_ylabel("Échantillons par classe")
ax.set_title("F1 (macro) selon la configuration")
for y in range(pivot.shape[0]):
    for x in range(pivot.shape[1]):
        ax.text(x, y, f"{pivot.values[y,x]:.2f}", ha="center", va="center", color="white")
fig.colorbar(im, ax=ax, label="F1")
plt.tight_layout(); plt.savefig("setfit_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
for e in sorted(df["epochs"].unique()):
    sub = df[df["epochs"]==e].sort_values("samples_per_class")
    ax.plot(sub["samples_per_class"], sub["f1"], marker="o", label=f"{e} epochs")
ax.set_xlabel("Échantillons par classe"); ax.set_ylabel("F1 (macro)")
ax.set_title("Effet du nombre d'échantillons sur le F1")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("setfit_curve.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Étapes finales

1. Télécharge `results_setfit.csv`, `setfit_heatmap.png`, `setfit_curve.png`.
2. Mets le code et les résultats sur **GitHub**.
3. Envoie le lien à ton enseignant.

Pour gagner en précision (au prix de la vitesse), remplace `BASE_MODEL` par `sentence-transformers/paraphrase-mpnet-base-v2` et relance la cellule 4.
